In [2]:
import numpy as np
import numpy.ma as ma
from admUI import admUI_numpy

from pid.optimizers import exact_gauss_tilde_pid, mmi_pid
from pid.utils.estimate import approx_pid_from_cov
from pid.utils.generate import sample_mult_poisson
from pid.utils.distributions import mult_poisson_dist

In [3]:
def get_params():
    lamda_m = np.array([2., 2.])
    lamda_x = np.array([1.])      # Noise to be added to X: shape (d_X,)
    lamda_y = np.array([1.])

    w_x1_vals = np.arange(11) / 10
    w_x2 = 0.5
    w_y1 = 0.5
    w_y2 = 0.5

    return lamda_m, lamda_x, lamda_y, w_x1_vals, w_x2, w_y1, w_y2

lamda_m, lamda_x, lamda_y, w_x1_vals, w_x2, w_y1, w_y2 = get_params()
print(w_x1_vals)

[0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]


In [4]:
dm = lamda_m.size  # Number of dimensions of M
dx = lamda_x.size
dy = lamda_y.size
print(f"dm: {dm}, dx: {dx}, dy: {dy}")

dm: 2, dx: 1, dy: 1


In [5]:
# We first compute the ground truth PID values for the multivariate Poisson distribution
MASK_THRESH = 1e-8

def pid(p, qstar, ind=1):
    """
    Computes the unique, redundant, synergistic and total information about M
    present in X_`ind` with respect to the other variables, given the optimal
    `q_star` from Bertschinger's definition.

    Here, `p` is the original joint distribution of all variables, where the
    first dimension of p represents M, and the remaining dimensions represent
    X_i, the variables being used in the decomposition.

    `q_star` may be computed using one of the two functions below,
    `compute_qstar` or `compute_qstar_admui`.

    Note: X_i's are 1-indexed, so X_1 is the first X variable. Equivalently,
          `ind` is the index of X_`ind` in *p*.
    """

    # Roll axes so that the variable of interest is always at index 1
    qstar = np.rollaxis(qstar, ind, 1)
    p = np.rollaxis(p, ind, 1)
    # Mask out near-zero values to avoid divide-by-zero and log errors
    qstar = ma.array(qstar, mask=(abs(qstar) < MASK_THRESH))
    p = ma.array(p, mask=(abs(p) < MASK_THRESH))

    # First, compute the unique information
    axes = (0, 1)
    qrest = ma.sum(qstar, axis=axes, keepdims=True)  # q*(x2)
    qstar_rest = qstar / qrest                       # q*(m, x1 | x2)

    qmrest = ma.sum(qstar, axis=1, keepdims=True)    # q*(m, x2)
    qm_rest = qmrest / qrest                         # q*(m | x2)

    qrestx = ma.sum(qstar, axis=0, keepdims=True)    # q*(x1, x2)
    qx_rest = qrestx / qrest                         # q*(x1 | x2)

    uniq_info = ma.sum(qstar * ma.log2(qstar_rest / (qm_rest * qx_rest)))

    # Then, compute redundant information
    axes = tuple(range(2, qstar.ndim))
    qmx = ma.sum(qstar, axis=axes, keepdims=True)    # q*(m, x1)
    axes = tuple(range(1, qstar.ndim))
    qm = ma.sum(qstar, axis=axes, keepdims=True)     # q*(m)
    axes = list(range(qstar.ndim))
    axes.remove(1)
    axes = tuple(axes)
    qx = ma.sum(qstar, axis=axes, keepdims=True)     # q*(x1)
    Imx = ma.sum(qmx * ma.log2(qmx / (qm * qx)))     # I(M ; X1)
    red_info = Imx - uniq_info

    # Alternate way to compute redundant info from p
    #axes = tuple(range(1, p.ndim))
    #pm = ma.sum(p, axis=axes, keepdims=True)
    #axes = tuple(range(2, p.ndim))
    #pmx = ma.sum(p, axis=axes, keepdims=True)
    #axes = list(range(p.ndim))
    #axes.remove(1)
    #axes = tuple(axes)
    #px = ma.sum(p, axis=axes, keepdims=True)
    #Imx = ma.sum(pmx * ma.log2(pmx / (pm * px)))
    #red_info = Imx - uniq_info

    # Finally, synergistic information
    axes = tuple(range(1, p.ndim))
    pm = ma.sum(p, axis=axes, keepdims=True)         # p(m)
    pall = ma.sum(p, axis=0, keepdims=True)          # p(x1, x2)
    Imall_p = ma.sum(p * ma.log2(p / (pm * pall)))   # I_p(M ; (X1, X2))
    Imall_q = ma.sum(qstar * ma.log2(qstar / (qm * qrestx))) # I_q(M ; (X1, X2))
    syn_info = Imall_p - Imall_q

    return (uniq_info, red_info, syn_info, Imall_p)


def compute_qstar_admui(p, M_dim, lamdas, ind=1, maxiter=250, verbose=False):
    """
    Compute the optimal distribution for the bivariate PID of information about
    M contained between X_ind and the other variables, using the iterative
    algorithm given by Banerjee et al. (ISIT 2018).

    Parameters:
        p - np.ndarry
            Joint distribution of (M, X1, X2, ...), each dimension of p
            represents these variables in order
        M_dim - int
                Should be equal to p.shape[0]
        lamdas - not used
        ind - int
              Index to be used as X1 (remaining indices are treated together
              as X2)
        verbose - not used
    """

    p = np.rollaxis(p, ind, 1)
    newshape = p.shape
    p = p.reshape((M_dim, p.shape[1], -1))  # Reshape into a bivariate PID

    pm = p.sum(axis=(1, 2))
    px_m = p.sum(axis=2).T / pm  # p(x1 | m)
    py_m = p.sum(axis=1).T / pm  # p(y | m) - here, `y` is the same as x2

    qstar = admUI_numpy.computeQUI_numpy(px_m, py_m, pm, maxiter=maxiter)
    # In the above, `maxiter` may need to be tuned based on performance

    qstar = qstar.reshape(newshape)  # Reshape back into the original shape
    qstar = np.rollaxis(qstar, 1, ind+1)  # And place X1 in its original index
    return qstar

def gt_pid(lamda_m, w_x, w_y, lamda_x, lamda_y, D):
    p = mult_poisson_dist(lamda_m, w_x, w_y, lamda_x, lamda_y, D=D)
    D = p.shape[0]
    # Compute Bertschinger's PID using the Banerjee et al. package
    p = p.reshape((D**2, D, D))
    qstar = compute_qstar_admui(p, p.shape[0], [], maxiter=1000)
    uix, ri, si, imxy = pid(p, qstar)  # UI corresponds to X
    uiy = imxy - uix - ri - si

    ret = (imxy, imxy, uix, uiy, ri, si)
    return ret

In [6]:
w_x = np.array([[0.5, 0.5]])  # Binomial thinning weights: shape (dx, dm)
w_y = np.array([[0.5, 0.5]])
m, x, y = sample_mult_poisson(100, lamda_m, w_x, w_y, lamda_x, lamda_y)
print(m.shape, x.shape, y.shape)

(2, 100) (1, 100) (1, 100)


In [7]:
n = 10000  # Sample size
D = 10    # Support of distribution in each dimension
pid_defn_names = ['tilde', 'delta', 'mmi']
pid_defns = [exact_gauss_tilde_pid, approx_pid_from_cov, mmi_pid]

for i, w_x1 in enumerate(w_x1_vals):
    print(f"\nw_x1: {w_x1}")
    w_x = np.array([[w_x1, w_x2]])  # Binomial thinning weights: shape (dx, dm)
    w_y = np.array([[w_y1, w_y2]])

    m, x, y = sample_mult_poisson(n, lamda_m, w_x, w_y, lamda_x, lamda_y)  # Shape: (d, n)
    mxy = np.vstack((m, x, y))
    cov = np.corrcoef(mxy)  # Shape: (dm+dx+dy, dm+dx+dy)

    # ground truth (time-consuming)
    # ret = gt_pid(lamda_m, w_x, w_y, lamda_x, lamda_y, D)
    # imxy, uix, uiy, ri, si = ret[2], *ret[-4:]
    # print(f"truth: imxy: {imxy}, uix: {uix}, uiy: {uiy}, ri: {ri}, si: {si}")

    for pid_name, pid_defn in zip(pid_defn_names, pid_defns):
        ret = pid_defn(cov, dm, dx, dy)
        imxy, uix, uiy, ri, si = ret[2], *ret[-4:]
        print(f"pid_name: {pid_name}, imxy: {imxy}, uix: {uix}, uiy: {uiy}, ri: {ri}, si: {si}")


w_x1: 0.0
pid_name: tilde, imxy: 0.46635975156429144, uix: 0.14103907661760673, uiy: 0.21774505262929492, ri: 0.0684483508278701, si: 0.03912727148951972
pid_name: delta, imxy: 0.46635978911876014, uix: 0.11436730898920706, uiy: 0.19107328500089504, ri: 0.09512011845626965, si: 0.06579907667238839
pid_name: mmi, imxy: 0.46635975156429127, uix: 0.0, uiy: 0.07670597601168819, ri: 0.2094874274454768, si: 0.18016634810712628

w_x1: 0.1
pid_name: tilde, imxy: 0.43561200645447534, uix: 0.10037784881748057, uiy: 0.19053684106183097, ri: 0.09122388705786533, si: 0.05347342951729844
pid_name: delta, imxy: 0.43561204101365125, uix: 0.07344663199004188, uiy: 0.163605624234392, ri: 0.11815510388530397, si: 0.08040468090391341
pid_name: mmi, imxy: 0.4356120064544755, uix: 0.0, uiy: 0.0901589922443504, ri: 0.19160173587534593, si: 0.15385127833477918

w_x1: 0.2
pid_name: tilde, imxy: 0.4415244719337357, uix: 0.05106655946812605, uiy: 0.1543122979532723, ri: 0.1456576394842014, si: 0.090487975028135